A basic function to fill 96 well deep well plates from a trough using the starlet. This is written to support sampling from pioreactors with more replicates. Should use one column of tips. 

In [8]:
#Star startup code (does something, needs to be here)
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
import asyncio
from typing import List, Iterator

from pylabrobot.liquid_handling import LiquidHandler
from pylabrobot.liquid_handling.backends import STARBackend
from pylabrobot.resources.hamilton import STARLetDeck, hamilton_mfx_carrier_L5_base, TIP_CAR_480_A00
from pylabrobot.resources.hamilton.mfx_modules import hamilton_mfx_plateholder_DWP_metal_tapped
#from pylabrobot.resources.opentrons.tube_racks import (
#    opentrons_24_tuberack_generic_1point5ml_snapcap_short,
#)
# from pylabrobot.resources.tube_adapter import TubeRackAdapter
from pylabrobot.resources.agenbio.plates import AGenBio_1_troughplate_100000uL_Fl
from pylabrobot.resources.bioer.plates import BioER_96_wellplate_Vb_2200uL
from pylabrobot.resources import (
    hamilton_96_tiprack_50uL_filter, # 50 µL filtered
    hamilton_96_tiprack_1000uL_filter,     # 1000 µL filtered 
    hamilton_96_tiprack_10uL_filter #Tip Rack with 96 10ul Low Volume Tip with filter
)

###############################################################################
# 0) build LiquidHandler + deck
###############################################################################
backend = STARBackend()
lh      = LiquidHandler(backend=backend, deck=STARLetDeck())
# await lh.stop()
await lh.setup(skip_autoload=True)

2026-03-02 11:34:19,954 - pylabrobot.io.usb - INFO - Finding USB device...
2026-03-02 11:34:19,955 - pylabrobot.io.usb - INFO - Found USB device.
2026-03-02 11:34:19,958 - pylabrobot.io.usb - INFO - Found endpoints. 
Write:
       ENDPOINT 0x2: Bulk OUT ===============================
       bLength          :    0x7 (7 bytes)
       bDescriptorType  :    0x5 Endpoint
       bEndpointAddress :    0x2 OUT
       bmAttributes     :    0x2 Bulk
       wMaxPacketSize   :   0x40 (64 bytes)
       bInterval        :    0x0 
Read:
       ENDPOINT 0x81: Bulk IN ===============================
       bLength          :    0x7 (7 bytes)
       bDescriptorType  :    0x5 Endpoint
       bEndpointAddress :   0x81 IN
       bmAttributes     :    0x2 Bulk
       wMaxPacketSize   :   0x40 (64 bytes)
       bInterval        :    0x0


2026-03-02 11:34:23,133 - pylabrobot - INFO - Running backend initialization procedure.


In [4]:
from pylabrobot.resources.plate import Plate
from pylabrobot.resources.utils import create_ordered_items_2d
from pylabrobot.resources.well import (
  CrossSectionType,
  Well,
  WellBottomType,
)

def VWR_96_wellplate_100_Vb_on_starCarrier_182070(name: str, with_lid: bool = False) -> Plate:
  """
This plate is a VWR PCR plate 96 well low-profile, half-skirted, ABI-FAST type plate.
VWR cat no. 89218-296
It is half-skirted so it must reside in another plate like a Cor_96_wellplate_360ul_Fb
It is currently on a STARLet carrier with catalog number 182070, and the plate is modeled on the carrier.
  """
  
  return Plate(
    name=name,
    size_x=127.55,
    size_y=85.0,
    size_z=18.6,
    # lid=lid,
    model=VWR_96_wellplate_100_Vb_on_starCarrier_182070.__name__,
    ordered_items=create_ordered_items_2d(
      Well,
      num_items_x=12,
      num_items_y=8,
      dx=11,  # measured
      dy=10,  # measured
      dz=2.15, # measured
      item_dx=9.0,
      item_dy=9.0,
      size_x=5.4,  # measured
      size_y=5.4,  # measured
      size_z=19, # measured well depth
      material_z_thickness=1.0,
      bottom_type=WellBottomType.V,
      cross_section_type=CrossSectionType.CIRCLE,
      max_volume=100,
    ),
  )

In [5]:
tip_car = TIP_CAR_480_A00("tip_car")
lh.deck.assign_child_resource(tip_car, rails=25)


tiprack_1000 = hamilton_96_tiprack_1000uL_filter("tips_00")
tiprack_50 = hamilton_96_tiprack_50uL_filter("tips_01")
tiprack_10 = hamilton_96_tiprack_10uL_filter("tips_02")
tip_car[0] = tiprack_1000
tip_car[1] = tiprack_50
tip_car[2] = tiprack_10

# Trough plate on rails=19, module slot 0
trough_module = hamilton_mfx_plateholder_DWP_metal_tapped("trough_module")
car_19 = hamilton_mfx_carrier_L5_base("car_19", modules={0:trough_module})
lh.deck.assign_child_resource(car_19, rails=19)
ab_trough = AGenBio_1_troughplate_100000uL_Fl("ab_trough")
trough_module.assign_child_resource(ab_trough)

# BioER DW plate rails=13, module slot 0
module_holding_dw_plate1 = hamilton_mfx_plateholder_DWP_metal_tapped("module_holding_dw_plate1")
module_holding_dw_plate2 = hamilton_mfx_plateholder_DWP_metal_tapped("module_holding_dw_plate2")
car_13 = hamilton_mfx_carrier_L5_base("car_13", modules={0:module_holding_dw_plate1,1:module_holding_dw_plate2})
lh.deck.assign_child_resource(car_13, rails=13)
dw_plate1 = BioER_96_wellplate_Vb_2200uL("dw_plate1")
module_holding_dw_plate1.assign_child_resource(dw_plate1)
dw_plate2 = BioER_96_wellplate_Vb_2200uL("dw_plate2")
module_holding_dw_plate2.assign_child_resource(dw_plate2)

In [6]:

async def fill_plate(plate):
    '''Fills a plate from a trough, a column at a time, 
    Needs tips in column 1 of the tip carrier
    Takes ~ 6.5 min to fill a plate
    Currently puts tips back into the tip rack, needs a helper to discard tips to trash after multipul plates are filled
    '''
    channels = [0,1,2,3,4,5,6,7]
    await lh.pick_up_tips(tiprack_1000[channels], use_channels=channels)
    for col in range(1, 13):
        await lh.aspirate(
            ab_trough["A1"] * 8,
            vols=[1000] * 8,
            use_channels=channels,
            liquid_height=[2] * 8,
            )
        await lh.dispense(
            plate[f"A{col}:H{col}"],
            vols=[1000] * 8,
            use_channels=channels,
            liquid_height=[20] * 8,
            blow_out=[1] * 8,
        )  
    await lh.drop_tips(tiprack_1000[channels], use_channels=channels)


async def trash_tips():
    
    channels = [0,1,2,3,4,5,6,7]
    try:
      await lh.discard_tips(use_channels=channels)
    except Exception:
      pass


In [ ]:
await fill_plate(dw_plate1)
await fill_plate(dw_plate2)




In [ ]:
await lh.stop()

<coroutine object Machine.stop at 0x7af87c4898a0>